# Image Feature Extraction **[DEMO]**

Determine which device to run PyTorch on:
- CUDA if an Nvidia GPU is installed
- CPU otherwise

In [ ]:
import torch

# Set device to CUDA if we have an Nvidia GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device {device}")

Load image datasets

In [ ]:
from PIL import Image
import requests

img_urls = [
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.png",
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.jpeg",
]
image_real = Image.open(requests.get(img_urls[0], stream=True).raw).convert("RGB")
image_gen = Image.open(requests.get(img_urls[1], stream=True).raw).convert("RGB")

cow1 = Image.open(r"demo_imgs/Cow1.jpg")
cow2 = Image.open(r"demo_imgs/Cow2.jpg")

Import pretrained image processor and ML model

In [ ]:
from transformers import AutoImageProcessor, AutoModel

processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = AutoModel.from_pretrained("google/vit-base-patch16-224").to(device)

Basic inference function

In [ ]:
def infer(image):
    inputs = processor(image, return_tensors="pt").to(device)
    outputs = model(**inputs)
    return outputs.pooler_output

Pass images to inference function to obtain embeddings

In [ ]:
embed_real = infer(image_real)
embed_gen = infer(image_gen)

embed_cow1 = infer(cow1)
embed_cow2 = infer(cow2)

Calculate similarity scores

In [ ]:
from torch.nn.functional import cosine_similarity

similarity_score = cosine_similarity(embed_real, embed_gen)
print(f"Similarity score: {similarity_score[0]:.4f}")

similarity_score = cosine_similarity(embed_real, embed_real)
print(f"Similarity score: {similarity_score[0]:.4f}")

similarity_score = cosine_similarity(embed_cow1, embed_cow2)
print(f"Similarity score: {similarity_score[0]:.4f}")

similarity_score = cosine_similarity(embed_real, embed_cow1)
print(f"Similarity score: {similarity_score[0]:.4f}")